# Лабораторная работа № 5

## Задача об эпидемии

Вариант 58.

N = 17854
I(0) = 199
R(0) = 35
S(0) = 17620

Исследуются два режима:

1. I(0) <= I* — заболевшие изолированы;
2. I(0) > I* — инфекция распространяется.

In [1]:
using DrWatson
@quickactivate "project"

using DifferentialEquations
using Plots
using DataFrames
using CSV

## Initial data

In [2]:
N = 17_854.0

I0 = 199.0
R0 = 35.0
S0 = N - I0 - R0

u0 = [S0, I0, R0]

alpha = 1.0e-5
beta = 0.02

tspan = (0.0, 120.0)
save_step = 0.1

mkpath("data")
mkpath("plots")

println("Population N = ", N)
println("S(0) = ", S0)
println("I(0) = ", I0)
println("R(0) = ", R0)
println("alpha = ", alpha)
println("beta = ", beta)

Population N = 17854.0
S(0) = 17620.0
I(0) = 199.0
R(0) = 35.0
alpha = 1.0e-5
beta = 0.02


## Case 1: I(0) <= I*

Infected individuals are isolated:

dS/dt = 0
dI/dt = -beta I
dR/dt = beta I

In [3]:
function epidemic_isolated!(du, u, p, t)
    S, I, R = u

    du[1] = 0.0
    du[2] = -beta * I
    du[3] = beta * I
end

problem1 = ODEProblem(
    epidemic_isolated!,
    u0,
    tspan,
)

solution1 = solve(
    problem1,
    Tsit5();
    saveat=save_step,
    abstol=1e-9,
    reltol=1e-9,
)

t1 = solution1.t
S1 = getindex.(solution1.u, 1)
I1 = getindex.(solution1.u, 2)
R1 = getindex.(solution1.u, 3)

1201-element Vector{Float64}:
  35.0
  35.39760226520072
  35.79441012054631
  36.190425153269956
  36.58564894742742
  36.98008308391564
  37.37372914047519
  37.76658869169108
  38.15866330901047
  38.549954560719776
  38.940464011975564
  39.33019322481885
  39.71914375817512
   ⋮
 215.5455630847816
 215.58243507431646
 215.61923339355718
 215.65595818970124
 215.69260960965244
 215.72918780002104
 215.76569290712368
 215.80212507698334
 215.83848445532948
 215.87477118759782
 215.91098541893058
 215.94712729417628

## Case 2: I(0) > I*

Infection spreads:

dS/dt = -alpha S I
dI/dt = alpha S I - beta I
dR/dt = beta I

In [4]:
function epidemic_active!(du, u, p, t)
    S, I, R = u

    infection = alpha * S * I
    recovery = beta * I

    du[1] = -infection
    du[2] = infection - recovery
    du[3] = recovery
end

problem2 = ODEProblem(
    epidemic_active!,
    u0,
    tspan,
)

solution2 = solve(
    problem2,
    Tsit5();
    saveat=save_step,
    abstol=1e-9,
    reltol=1e-9,
)

t2 = solution2.t
S2 = getindex.(solution2.u, 1)
I2 = getindex.(solution2.u, 2)
R2 = getindex.(solution2.u, 3)

1201-element Vector{Float64}:
    35.0
    35.4011243914586
    35.80856209227412
    36.22241098270043
    36.642770413414276
    37.069741225855466
    37.50342577371419
    37.94392794290758
    38.391353174479384
    38.84580848394286
    39.30740248602426
    39.7762454134383
    40.25244914183426
     ⋮
 14818.254225700028
 14824.297982612492
 14830.329729403335
 14836.349489743601
 14842.357287258857
 14848.3531455292
 14854.337088089243
 14860.309138428138
 14866.269319989553
 14872.217656171684
 14878.154170327305
 14884.07888576704

## Save results

In [5]:
CSV.write(
    "data/epidemic_isolated.csv",
    DataFrame(
        time=t1,
        susceptible=S1,
        infected=I1,
        recovered=R1,
    ),
)

CSV.write(
    "data/epidemic_active.csv",
    DataFrame(
        time=t2,
        susceptible=S2,
        infected=I2,
        recovered=R2,
    ),
)

"data/epidemic_active.csv"

## Case 1 plot

In [6]:
p1 = plot(
    t1,
    S1;
    label="S(t)",
    linewidth=2,
    xlabel="Time",
    ylabel="Population",
    title="Case 1: I(0) <= I*",
)

plot!(p1, t1, I1; label="I(t)", linewidth=2)
plot!(p1, t1, R1; label="R(t)", linewidth=2)

savefig(
    p1,
    "plots/epidemic_isolated.png",
)

"/home/adsvettsova/work/study/2026-1/2026-1--study-mathematical-modeling/labs/lab05/project/notebooks/01_epidemic/plots/epidemic_isolated.png"

## Case 2 plot

In [7]:
p2 = plot(
    t2,
    S2;
    label="S(t)",
    linewidth=2,
    xlabel="Time",
    ylabel="Population",
    title="Case 2: I(0) > I*",
)

plot!(p2, t2, I2; label="I(t)", linewidth=2)
plot!(p2, t2, R2; label="R(t)", linewidth=2)

savefig(
    p2,
    "plots/epidemic_active.png",
)

"/home/adsvettsova/work/study/2026-1/2026-1--study-mathematical-modeling/labs/lab05/project/notebooks/01_epidemic/plots/epidemic_active.png"

## Infected comparison

In [8]:
p3 = plot(
    t1,
    I1;
    label="I(0) <= I*",
    linewidth=2,
    xlabel="Time",
    ylabel="Infected",
    title="Comparison of epidemic regimes",
)

plot!(
    p3,
    t2,
    I2;
    label="I(0) > I*",
    linewidth=2,
)

savefig(
    p3,
    "plots/epidemic_comparison.png",
)

peak_index = argmax(I2)

println()
println("Case 1 final:")
println("S = ", last(S1))
println("I = ", last(I1))
println("R = ", last(R1))

println()
println("Case 2:")
println("Peak infected = ", I2[peak_index])
println("Peak time = ", t2[peak_index])
println("Final S = ", last(S2))
println("Final I = ", last(I2))
println("Final R = ", last(R2))

println()
println("Population check, case 1 = ", last(S1 + I1 + R1))
println("Population check, case 2 = ", last(S2 + I2 + R2))

println()
println("LAB05 BASE OK")


Case 1 final:
S = 17620.0
I = 18.052872705823795
R = 215.94712729417628

Case 2:
Peak infected = 11467.203272408859
Peak time = 42.6
Final S = 10.509193620679092
Final I = 2959.4119206122737
Final R = 14884.07888576704

Population check, case 1 = 17854.0
Population check, case 2 = 17853.999999999993

LAB05 BASE OK
